# 04 -- Attention Entropy Analysis

Per-head entropy analysis: classify attention heads as S1-specialized,
S2-specialized, mixed, or non-specialized using the multi-metric consensus
framework (Fartale et al. 2025).

**What this notebook does:**
1. Load precomputed attention metrics from HDF5
2. Per-head Mann-Whitney U tests (conflict vs control)
3. BH-FDR correction across all heads/metrics
4. Multi-metric consensus classification
5. Layer-level summary heatmap
6. KV-group aggregate (more conservative for GQA)
7. Cross-model comparison of head specialization

**Modes:**
- **Synthetic** (default): planted differential at (layer 1, head 0)
- **Real**: switch `DATA_PATH` for real attention metrics

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

%matplotlib inline

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from s1s2.utils.io import (
    open_activations, load_problem_metadata, list_models,
    model_metadata, get_attention_metric, position_labels, position_valid,
)
from s1s2.attention.core import (
    METRIC_NAMES, METRIC_DIRECTIONS, AttentionConfig,
    bh_fdr_joint,
)
from s1s2.attention.heads import (
    run_all_head_differential_tests,
    classify_heads,
    head_classifications_to_records,
    HeadClassification,
)
from s1s2.viz.theme import set_paper_theme, COLORS

set_paper_theme()

In [ ]:
# ---- Mode selection ----
USE_SYNTHETIC = True  # Set to False for real data

if USE_SYNTHETIC:
    from tests.conftest import (
        build_synthetic_hdf5, SYNTH_MODEL_KEY,
        SYNTH_N_LAYERS, SYNTH_N_HEADS, SYNTH_N_KV_HEADS,
    )
    DATA_PATH = REPO_ROOT / "data" / "activations" / "synthetic_nb04.h5"
    build_synthetic_hdf5(DATA_PATH)
    MODEL_KEY = SYNTH_MODEL_KEY
    print(f"Built synthetic HDF5 at {DATA_PATH}")
    print(f"Planted entropy differential at (layer 1, head 0).")
else:
    DATA_PATH = REPO_ROOT / "data" / "activations" / "main.h5"  # REQUIRES REAL DATA
    MODEL_KEY = "meta-llama_Llama-3.1-8B-Instruct"  # REQUIRES REAL DATA
    print(f"Using real data at {DATA_PATH}")

In [ ]:
# --- Load attention metrics ---
with open_activations(DATA_PATH) as f:
    meta = load_problem_metadata(f)
    mm = model_metadata(f, MODEL_KEY)
    pos_labels = position_labels(f, MODEL_KEY)
    pos_valid = position_valid(f, MODEL_KEY)

    # Load all metrics
    attention_metrics = {}
    for metric_name in METRIC_NAMES:
        attention_metrics[metric_name] = get_attention_metric(f, MODEL_KEY, metric_name)

conflict = meta["conflict"]
n_layers = int(mm["n_layers"])
n_heads = int(mm["n_heads"])
n_kv_heads = int(mm["n_kv_heads"])

# Determine valid positions
valid_positions = []
valid_indices = []
for i, p in enumerate(pos_labels):
    if pos_valid[:, i].any():
        valid_positions.append(p)
        valid_indices.append(i)

print(f"Model: {MODEL_KEY}")
print(f"Architecture: {n_layers} layers, {n_heads} query heads, {n_kv_heads} KV heads")
print(f"GQA group size: {n_heads // n_kv_heads}")
print(f"Problems: {len(conflict)} (conflict={conflict.sum()}, control={(~conflict).sum()})")
print(f"Valid positions: {valid_positions}")
print(f"Attention metric shape: {attention_metrics['entropy'].shape}")

## Per-Head Differential Tests

Mann-Whitney U test at every (layer, head, position, metric) comparing conflict vs control.

In [ ]:
from s1s2.attention.core import ModelAttentionData

# Subset metrics to valid positions only
metrics_subset = {}
for name, arr in attention_metrics.items():
    metrics_subset[name] = arr[..., valid_indices].astype(np.float32)

# Build the data container
att_data = ModelAttentionData(
    model_key=MODEL_KEY,
    model_config_key=MODEL_KEY,
    family="synthetic",
    n_layers=n_layers,
    n_heads=n_heads,
    n_kv_heads=n_kv_heads,
    is_reasoning_model=bool(mm.get("is_reasoning_model", False)),
    position_labels=pos_labels,
    selected_positions=valid_positions,
    metrics=metrics_subset,
    conflict=conflict,
)

# Run all per-head tests
df_tests = run_all_head_differential_tests(att_data, metrics=METRIC_NAMES)
print(f"Total tests: {len(df_tests)}")
print(f"Columns: {list(df_tests.columns)}")

In [ ]:
# Apply BH-FDR
if not df_tests.empty:
    pvals = df_tests["p_value"].to_numpy(dtype=np.float64)
    rejected, qvals = bh_fdr_joint(pvals, q=0.05)
    df_tests["q_value"] = qvals
    df_tests["significant"] = rejected

    n_sig = df_tests["significant"].sum()
    print(f"Significant tests (q<0.05): {n_sig}/{len(df_tests)}")
    print(f"\nBy metric:")
    for m in METRIC_NAMES:
        sub = df_tests[df_tests["metric"] == m]
        n_m_sig = sub["significant"].sum()
        print(f"  {m}: {n_m_sig}/{len(sub)} significant")
else:
    print("No tests to analyze.")

## Head Classification

Apply the 3-of-5 multi-metric consensus rule: a head is S2-specialized if >= 3 metrics
are significant, the entropy effect size is >= 0.3, and all significant metrics point
in the same direction.

In [ ]:
classifs = classify_heads(
    df_tests,
    n_layers=n_layers,
    n_heads=n_heads,
    min_significant=3,
    entropy_effect_threshold=0.3,
)

# Summarize
from collections import Counter
class_counts = Counter(c.classification for c in classifs)
total_heads = n_layers * n_heads
print(f"Head classification summary ({total_heads} heads):")
for cls, count in sorted(class_counts.items()):
    print(f"  {cls}: {count} ({count / max(total_heads, 1):.1%})")

In [ ]:
# Build classification heatmap
class_map = {"non_specialized": 0, "S1_specialized": 1, "S2_specialized": 2, "mixed": 3}
class_colors = [COLORS["non_significant"], COLORS["s1"], COLORS["s2"], COLORS["baseline"]]
cmap = ListedColormap(class_colors)

grid = np.zeros((n_layers, n_heads), dtype=int)
for c in classifs:
    grid[c.layer, c.head] = class_map.get(c.classification, 0)

fig, ax = plt.subplots(figsize=(max(6, n_heads * 0.8), max(4, n_layers * 0.6)))
im = ax.imshow(grid, cmap=cmap, aspect="auto", vmin=0, vmax=3)
ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_title(f"Head classification: {MODEL_KEY}")
ax.set_xticks(range(n_heads))
ax.set_yticks(range(n_layers))

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS["non_significant"], label="Non-specialized"),
    Patch(facecolor=COLORS["s1"], label="S1-specialized"),
    Patch(facecolor=COLORS["s2"], label="S2-specialized"),
    Patch(facecolor=COLORS["baseline"], label="Mixed"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## Layer-Level Entropy Differential

Mean entropy differential (conflict - control) across heads at each layer.

In [ ]:
# Compute mean entropy differential per layer (at first valid position)
entropy = metrics_subset["entropy"]  # (n_problems, n_layers, n_heads, n_positions)
# Use first valid position
ent_pos0 = entropy[..., 0]  # (n_problems, n_layers, n_heads)

delta_per_head = (
    ent_pos0[conflict].mean(axis=0) - ent_pos0[~conflict].mean(axis=0)
)  # (n_layers, n_heads)

delta_per_layer = delta_per_head.mean(axis=1)  # (n_layers,)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: mean delta per layer
colors_bar = [COLORS["s2"] if d > 0 else COLORS["s1"] for d in delta_per_layer]
axes[0].bar(range(n_layers), delta_per_layer, color=colors_bar, edgecolor="white")
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Mean entropy delta (conflict - control)")
axes[0].set_title("Layer-level entropy differential")
axes[0].axhline(0, color="gray", linestyle="-", linewidth=0.5)

# Heatmap: delta per head
vmax = max(abs(delta_per_head.min()), abs(delta_per_head.max()), 0.01)
im = axes[1].imshow(delta_per_head, cmap="RdBu_r", aspect="auto",
                      vmin=-vmax, vmax=vmax)
axes[1].set_xlabel("Head")
axes[1].set_ylabel("Layer")
axes[1].set_title("Per-head entropy delta")
plt.colorbar(im, ax=axes[1], label="Delta (conflict - control)")

plt.tight_layout()
plt.show()

## Effect Size Distribution

In [ ]:
if not df_tests.empty:
    entropy_tests = df_tests[df_tests["metric"] == "entropy"]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Rank-biserial distribution
    axes[0].hist(entropy_tests["effect_size_rb"], bins=30,
                 color=COLORS["standard"], edgecolor="white", alpha=0.8)
    axes[0].axvline(0, color="gray", linestyle="-", linewidth=0.5)
    axes[0].axvline(0.3, color="red", linestyle="--", alpha=0.5, label="|r|=0.3")
    axes[0].axvline(-0.3, color="red", linestyle="--", alpha=0.5)
    axes[0].set_xlabel("Rank-biserial correlation")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Entropy effect size distribution")
    axes[0].legend(fontsize=8)

    # p-value distribution (should be uniform under null)
    axes[1].hist(entropy_tests["p_value"], bins=20,
                 color=COLORS["standard"], edgecolor="white", alpha=0.8)
    axes[1].axhline(len(entropy_tests) / 20, color="red", linestyle="--",
                     alpha=0.5, label="Uniform expectation")
    axes[1].set_xlabel("p-value")
    axes[1].set_ylabel("Count")
    axes[1].set_title("p-value distribution (entropy)")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## Cross-Model Comparison

Compare head specialization patterns between standard and reasoning models.
In real data mode, this compares e.g. Llama-3.1-8B-Instruct vs R1-Distill-Llama-8B.

In [ ]:
with open_activations(DATA_PATH) as f:
    all_models = list_models(f)

if len(all_models) > 1:
    print(f"Multiple models available: {all_models}")
    print("Cross-model comparison would overlay classification heatmaps here.")
    # REQUIRES REAL DATA: repeat the analysis for each model and compare
else:
    print(f"Only one model in dataset: {all_models}")
    print("Cross-model comparison requires multi-model extraction.")
    print("\nExpected pattern with real data:")
    print("  - Reasoning models (R1-Distill) should have MORE S2-specialized heads")
    print("  - Matched-architecture pairs (Llama vs R1-Distill-Llama) enable")
    print("    direct per-head comparison of how distillation changes specialization")

## Summary

**Key findings from this notebook:**

1. **Head specialization**: How many heads show significant conflict/control differentiation across multiple metrics?
2. **Layer localization**: Which layers concentrate the S1/S2-specialized heads?
3. **GQA caution**: Heads within the same KV group share key/value projections and are not independent. The KV-group analysis is the conservative report.
4. **Cross-model hypothesis**: reasoning training (distillation) should shift heads toward S2-specialization

**Caveats:**
- Gemma-2 alternates full/sliding-window layers -- analyze separately
- Always normalize entropy by `log2(t)` when comparing across sequence lengths
- GQA non-independence: report at both per-head and per-KV-group granularity